# Gabarito - Exercícios da Aula 01

**Módulo 02 - Fundamentos de Python para Bioinformática**

---

## Exercício 1: Análise de Composição de DNA

### Enunciado
Dada a sequência de DNA abaixo, calcule:
a) Frequência de cada base
b) Conteúdo GC (%)
c) Complemento reverso
d) Identifique se há códons de parada (TAA, TAG, TGA) na sequência

In [ ]:
# Sequência para análise
sequencia = 'ATGGCTAGCTAATCGATCGATCGATCGTAGCTAGCTAGCTAA'

# a) Frequência de cada base
print('=== a) Frequência de Bases ===')
for base in ['A', 'T', 'C', 'G']:
    contagem = sequencia.count(base)
    freq = contagem / len(sequencia) * 100
    print(f'  {base}: {contagem} ({freq:.1f}%)')

# b) Conteúdo GC
gc_count = sequencia.count('G') + sequencia.count('C')
gc_content = gc_count / len(sequencia) * 100
print(f'\n=== b) Conteúdo GC ===')
print(f'  GC: {gc_content:.1f}%')

# c) Complemento reverso
complemento = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}
comp_reverso = ''.join(complemento[base] for base in reversed(sequencia))
print(f'\n=== c) Complemento Reverso ===')
print(f'  Original:       {sequencia}')
print(f'  Comp. Reverso:  {comp_reverso}')

# d) Códons de parada
codons_parada = ['TAA', 'TAG', 'TGA']
encontrados = []
for i in range(0, len(sequencia) - 2, 3):
    codon = sequencia[i:i+3]
    if codon in codons_parada:
        encontrados.append((i, codon))

print(f'\n=== d) Códons de Parada ===')
if encontrados:
    for pos, codon in encontrados:
        print(f'  Posição {pos}: {codon}')
else:
    print('  Nenhum códon de parada encontrado no frame 0')

### Explicação
- **Conteúdo GC** influencia a temperatura de melting (Tm) de primers e amplicons
- **Complemento reverso** é essencial para design de primer reverse em PCR
- **Códons de parada** (TAA, TAG, TGA) sinalizam término de tradução; sua presença prematura pode indicar pseudogenes ou mutações

## Exercício 2: Comparação de Perfis STR

### Enunciado
Compare dois perfis genéticos e identifique marcadores comuns, alelos compartilhados e proporção de match.

In [ ]:
perfil_a = {
    'D3S1358': [15, 17],
    'TH01': [7, 9.3],
    'D21S11': [28, 30],
    'D18S51': [14, 17]
}

perfil_b = {
    'D3S1358': [15, 16],
    'TH01': [7, 9],
    'D21S11': [28, 30],
    'D18S51': [14, 15]
}

# a) Marcadores em comum
marcadores_comuns = set(perfil_a.keys()) & set(perfil_b.keys())
print(f'=== a) Marcadores em Comum ===')
print(f'  {sorted(marcadores_comuns)}')

# b) Alelos compartilhados por marcador
print(f'\n=== b) Alelos Compartilhados ===')
total_alelos_comparados = 0
total_alelos_compartilhados = 0

for marcador in sorted(marcadores_comuns):
    alelos_a = set(perfil_a[marcador])
    alelos_b = set(perfil_b[marcador])
    compartilhados = alelos_a & alelos_b
    
    total_alelos_comparados += 2  # diploide
    total_alelos_compartilhados += len(compartilhados)
    
    status = '✓ MATCH' if alelos_a == alelos_b else ('~ PARCIAL' if compartilhados else '✗ DIFERENTE')
    print(f'  {marcador}: A={sorted(alelos_a)} | B={sorted(alelos_b)} → {status} {sorted(compartilhados) if compartilhados else "[]"}')

# c) Proporção de alelos compartilhados
match_proportion = total_alelos_compartilhados / total_alelos_comparados
print(f'\n=== c) Match Proportion ===')
print(f'  Alelos compartilhados: {total_alelos_compartilhados}/{total_alelos_comparados}')
print(f'  Proporção: {match_proportion:.2%}')

# Interpretação forense
if match_proportion == 1.0:
    print('  → PERFIS IDÊNTICOS nos loci comparados')
elif match_proportion >= 0.5:
    print('  → Compatibilidade parcial (possível parentesco ou mistura)')
else:
    print('  → Perfis distintos (exclusão)')

### Explicação
- **Operações de conjuntos** (`&`, `|`, `-`) são ideais para comparação de perfis alélicos
- Em forense, **match proportion** é um indicador preliminar; cálculos formais usam Likelihood Ratios
- Alelos compartilhados parcialmente podem indicar **misturas de DNA** ou **parentesco biológico**

## Exercício 3: Processamento de Amostras com Pandas

### Enunciado
Usando o DataFrame de amostras forenses:
a) Filtre amostras com concentração > 10 ng/µL E não degradadas
b) Agrupe por tipo de amostra e calcule estatísticas de concentração
c) Crie uma nova coluna 'status' combinando degradação e qualidade

In [ ]:
import pandas as pd

# Recriar DataFrame
dados = {
    'id': ['AM001', 'AM002', 'AM003', 'AM004', 'AM005'],
    'tipo': ['sangue', 'saliva', 'tecido', 'sangue', 'osso'],
    'dna_concentracao_ng_ul': [25.4, 18.7, 42.1, 8.3, 3.2],
    'ratio_260_280': [1.82, 1.79, 1.85, 1.65, 1.45],
    'degradado': [False, False, False, True, True],
}
df = pd.DataFrame(dados)

# a) Filtrar amostras boas
amostras_boas = df[(df['dna_concentracao_ng_ul'] > 10) & (df['degradado'] == False)]
print('=== a) Amostras com conc > 10 ng/µL e não degradadas ===')
display(amostras_boas)

# b) Estatísticas por tipo
stats_por_tipo = df.groupby('tipo')['dna_concentracao_ng_ul'].agg(
    media='mean',
    mediana='median',
    minimo='min',
    maximo='max',
    n='count'
).round(2)
print('\n=== b) Estatísticas de Concentração por Tipo ===')
display(stats_por_tipo)

# c) Coluna status combinada
def classificar_status(row):
    if row['degradado']:
        return 'CRÍTICA - Degradada'
    elif row['ratio_260_280'] >= 1.8 and row['dna_concentracao_ng_ul'] >= 20:
        return 'EXCELENTE'
    elif row['ratio_260_280'] >= 1.6 and row['dna_concentracao_ng_ul'] >= 10:
        return 'ADEQUADA'
    else:
        return 'ATENÇÃO - Baixa qualidade'

df['status'] = df.apply(classificar_status, axis=1)
print('\n=== c) Classificação de Status ===')
display(df[['id', 'tipo', 'dna_concentracao_ng_ul', 'ratio_260_280', 'degradado', 'status']])

### Explicação
- **Filtragem booleana** com `&` (AND) e `|` (OR) é fundamental para triagem de amostras
- `groupby().agg()` permite múltiplas agregações simultâneas para relatórios QC
- `apply()` com função customizada permite lógica complexa de classificação baseada em múltiplas colunas
- Em laboratórios forenses, essa classificação determina a **estratégia de análise** (PCR padrão vs mini-STR vs NGS)

---

## Conceitos-Chave Revisados

| Conceito Python | Aplicação Forense |
|-----------------|-------------------|
| Strings e slicing | Manipulação de sequências DNA/RNA |
| Listas e list comprehension | Coleções de alelos, filtragem de dados |
| Dicionários | Perfis genéticos estruturados |
| Conjuntos (sets) | Comparação de perfis, interseção de alelos |
| Pandas DataFrame | Dados tabulares de amostras |
| groupby + agg | Relatórios QC por categoria |
| apply() | Classificação customizada de amostras |

---

> ⚖️ **Nota Ética**: Todos os dados são simulados para fins educacionais. Em análises reais, siga protocolos institucionais e normas SWGDAM/ISO 17025.